# M11 Deep Learning Architectures — Code Only

This Google Colab notebook contains only the model code and student-friendly comments.  
Charts, confusion matrices, evaluation tables, and written reflections have been removed.


## Setup and Imports

In [ ]:
# Imports TensorFlow so the notebook can build and train neural networks.
import tensorflow as tf

# Imports Keras from TensorFlow for creating the models.
from tensorflow import keras

# Imports common neural-network layers used throughout the assignment.
from tensorflow.keras import layers

# Imports Sequential so layers can be arranged in order.
from tensorflow.keras.models import Sequential

# Imports NumPy for numerical operations and array handling.
import numpy as np


## Part A: Multilayer Perceptron Using MNIST

In [ ]:
# Loads the MNIST handwritten-digit training and testing datasets.
(X_train_mnist, y_train_mnist), (X_test_mnist, y_test_mnist) = keras.datasets.mnist.load_data()

# Converts the training images to decimal values and scales pixels between 0 and 1.
X_train_mnist = X_train_mnist.astype("float32") / 255.0

# Converts the testing images to decimal values and scales pixels between 0 and 1.
X_test_mnist = X_test_mnist.astype("float32") / 255.0

# Flattens each 28-by-28 training image into a one-dimensional list of 784 pixel values.
X_train_mnist_flat = X_train_mnist.reshape(-1, 28 * 28)

# Flattens each 28-by-28 testing image into a one-dimensional list of 784 pixel values.
X_test_mnist_flat = X_test_mnist.reshape(-1, 28 * 28)

# Creates a simple multilayer perceptron model.
mlp_model = Sequential([
    # Defines the expected input shape of 784 pixel values.
    layers.Input(shape=(784,)),

    # Adds a hidden layer with 128 neurons and the ReLU activation function.
    layers.Dense(128, activation="relu"),

    # Adds dropout to reduce the chance that the model memorizes the training data.
    layers.Dropout(0.20),

    # Adds an output layer with one probability for each digit from 0 through 9.
    layers.Dense(10, activation="softmax")
])

# Configures the MLP with Adam, a multiclass loss function, and accuracy tracking.
mlp_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Trains the MLP on the MNIST training images.
mlp_model.fit(
    X_train_mnist_flat,
    y_train_mnist,
    epochs=5,
    batch_size=128,
    validation_split=0.10
)

# Uses the trained MLP to create class probabilities for the testing images.
mlp_probabilities = mlp_model.predict(X_test_mnist_flat)

# Converts the probabilities into predicted digit labels.
mlp_predictions = np.argmax(mlp_probabilities, axis=1)


## Part B: Convolutional Neural Network Using Fashion-MNIST

In [ ]:
# Loads the Fashion-MNIST clothing-image training and testing datasets.
(X_train_fashion, y_train_fashion), (X_test_fashion, y_test_fashion) = keras.datasets.fashion_mnist.load_data()

# Converts the training images to decimal values and scales pixels between 0 and 1.
X_train_fashion = X_train_fashion.astype("float32") / 255.0

# Converts the testing images to decimal values and scales pixels between 0 and 1.
X_test_fashion = X_test_fashion.astype("float32") / 255.0

# Adds a channel dimension so the training images can be processed by convolutional layers.
X_train_fashion_cnn = X_train_fashion.reshape(-1, 28, 28, 1)

# Adds a channel dimension so the testing images can be processed by convolutional layers.
X_test_fashion_cnn = X_test_fashion.reshape(-1, 28, 28, 1)

# Stores readable names for the ten Fashion-MNIST categories.
fashion_class_names = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

# Creates a convolutional neural network for classifying clothing images.
cnn_model = Sequential([
    # Defines the expected shape of each grayscale image.
    layers.Input(shape=(28, 28, 1)),

    # Uses 32 filters to learn basic visual features such as edges and curves.
    layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),

    # Reduces the image dimensions while keeping important learned features.
    layers.MaxPooling2D(pool_size=(2, 2)),

    # Uses 64 filters to learn more detailed visual patterns.
    layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),

    # Reduces the feature-map dimensions a second time.
    layers.MaxPooling2D(pool_size=(2, 2)),

    # Converts the two-dimensional feature maps into a one-dimensional vector.
    layers.Flatten(),

    # Adds a fully connected hidden layer for learning category relationships.
    layers.Dense(128, activation="relu"),

    # Adds dropout to help reduce overfitting.
    layers.Dropout(0.30),

    # Produces one probability for each Fashion-MNIST clothing category.
    layers.Dense(10, activation="softmax")
])

# Configures the CNN for multiclass image classification.
cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Trains the CNN on the Fashion-MNIST training images.
cnn_model.fit(
    X_train_fashion_cnn,
    y_train_fashion,
    epochs=5,
    batch_size=128,
    validation_split=0.10
)

# Uses the trained CNN to create category probabilities for the testing images.
cnn_probabilities = cnn_model.predict(X_test_fashion_cnn)

# Converts the probabilities into predicted clothing-category numbers.
cnn_predictions = np.argmax(cnn_probabilities, axis=1)


## Part C: Recurrent Neural Network Using IMDB Reviews

In [ ]:
# Sets the maximum number of common IMDB words kept in the vocabulary.
vocabulary_size = 10000

# Sets the maximum number of words allowed in each review.
maximum_review_length = 200

# Loads the IMDB movie-review training and testing datasets.
(X_train_imdb, y_train_imdb), (X_test_imdb, y_test_imdb) = keras.datasets.imdb.load_data(
    num_words=vocabulary_size
)

# Pads or shortens every training review to the same length.
X_train_imdb = keras.utils.pad_sequences(
    X_train_imdb,
    maxlen=maximum_review_length
)

# Pads or shortens every testing review to the same length.
X_test_imdb = keras.utils.pad_sequences(
    X_test_imdb,
    maxlen=maximum_review_length
)

# Creates an LSTM-based recurrent neural network for sentiment classification.
rnn_model = Sequential([
    # Defines the expected input as a sequence of 200 word numbers.
    layers.Input(shape=(maximum_review_length,)),

    # Converts each word number into a learned vector with 64 values.
    layers.Embedding(
        input_dim=vocabulary_size,
        output_dim=64
    ),

    # Reads the review in sequence and remembers useful information from earlier words.
    layers.LSTM(64),

    # Adds dropout to reduce overfitting.
    layers.Dropout(0.30),

    # Produces a probability between 0 and 1 for negative or positive sentiment.
    layers.Dense(1, activation="sigmoid")
])

# Configures the RNN for binary sentiment classification.
rnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Trains the RNN on the IMDB movie reviews.
rnn_model.fit(
    X_train_imdb,
    y_train_imdb,
    epochs=3,
    batch_size=128,
    validation_split=0.10
)

# Uses the trained RNN to create sentiment probabilities for the testing reviews.
rnn_probabilities = rnn_model.predict(X_test_imdb)

# Converts probabilities of 0.5 or greater into positive-review predictions.
rnn_predictions = (rnn_probabilities >= 0.50).astype("int32")


### Upload and Predict Reviews from the Text File

In [ ]:
# Imports the Google Colab file-upload tool.
from google.colab import files

# Opens a browser window so the IMDB review text file can be uploaded.
uploaded_files = files.upload()

# Stores the expected name of the uploaded text file.
review_file_name = "imdb_sample_reviews.txt"

# Opens the uploaded text file for reading.
with open(review_file_name, "r", encoding="utf-8") as review_file:
    # Reads each nonempty line as a separate movie review.
    custom_reviews = [
        line.strip()
        for line in review_file
        if line.strip()
    ]

# Gets Keras's original IMDB word-to-number dictionary.
imdb_word_index = keras.datasets.imdb.get_word_index()

# Adds three because the IMDB dataset reserves its first few numbers for special tokens.
imdb_word_index = {
    word: index + 3
    for word, index in imdb_word_index.items()
}

# Assigns the special padding token.
imdb_word_index["<PAD>"] = 0

# Assigns the special beginning-of-review token.
imdb_word_index["<START>"] = 1

# Assigns the special unknown-word token.
imdb_word_index["<UNK>"] = 2

# Assigns the special unused token.
imdb_word_index["<UNUSED>"] = 3

# Defines a function that converts regular review text into IMDB word numbers.
def encode_review(review_text):
    # Converts the review to lowercase and separates it into individual words.
    review_words = review_text.lower().split()

    # Converts each word into its vocabulary number or the unknown-word number.
    encoded_words = [
        imdb_word_index.get(word, 2)
        for word in review_words
    ]

    # Adds the beginning-of-review token before the encoded words.
    return [1] + encoded_words

# Encodes every uploaded movie review.
encoded_custom_reviews = [
    encode_review(review)
    for review in custom_reviews
]

# Pads or shortens every custom review to the same length used during training.
padded_custom_reviews = keras.utils.pad_sequences(
    encoded_custom_reviews,
    maxlen=maximum_review_length
)

# Uses the RNN to predict sentiment probabilities for the uploaded reviews.
custom_review_probabilities = rnn_model.predict(padded_custom_reviews)

# Converts the probabilities into readable positive or negative labels.
custom_review_labels = [
    "Positive" if probability[0] >= 0.50 else "Negative"
    for probability in custom_review_probabilities
]


## Part D: DCGAN Using MNIST

In [ ]:
# Selects the first 10,000 MNIST training images so the demonstration trains faster.
dcgan_images = X_train_mnist[:10000]

# Reshapes the images so each one has a grayscale channel.
dcgan_images = dcgan_images.reshape(-1, 28, 28, 1)

# Rescales the images from 0-to-1 values into negative-1-to-positive-1 values.
dcgan_images = (dcgan_images * 2.0) - 1.0

# Sets the number of images processed together during training.
dcgan_batch_size = 128

# Creates a TensorFlow dataset from the MNIST images.
dcgan_dataset = tf.data.Dataset.from_tensor_slices(dcgan_images)

# Shuffles the images before each training pass.
dcgan_dataset = dcgan_dataset.shuffle(buffer_size=10000)

# Groups images into training batches.
dcgan_dataset = dcgan_dataset.batch(dcgan_batch_size)

# Sets the size of the random-noise vector used by the generator.
noise_dimension = 100

# Creates the generator network that turns random noise into synthetic digit images.
generator = Sequential([
    # Accepts a random-noise vector with 100 values.
    layers.Input(shape=(noise_dimension,)),

    # Projects the noise into enough values for a 7-by-7 feature map with 256 channels.
    layers.Dense(7 * 7 * 256, use_bias=False),

    # Normalizes the generator's learned values.
    layers.BatchNormalization(),

    # Applies a ReLU-style activation that also allows small negative values.
    layers.LeakyReLU(),

    # Reshapes the vector into a small stack of feature maps.
    layers.Reshape((7, 7, 256)),

    # Enlarges the feature maps from 7-by-7 to 14-by-14.
    layers.Conv2DTranspose(
        128,
        kernel_size=(5, 5),
        strides=(1, 1),
        padding="same",
        use_bias=False
    ),

    # Normalizes the enlarged feature maps.
    layers.BatchNormalization(),

    # Applies another LeakyReLU activation.
    layers.LeakyReLU(),

    # Enlarges the feature maps from 14-by-14 to 28-by-28.
    layers.Conv2DTranspose(
        64,
        kernel_size=(5, 5),
        strides=(2, 2),
        padding="same",
        use_bias=False
    ),

    # Normalizes the feature maps.
    layers.BatchNormalization(),

    # Applies another LeakyReLU activation.
    layers.LeakyReLU(),

    # Produces one final grayscale image with values between negative 1 and positive 1.
    layers.Conv2DTranspose(
        1,
        kernel_size=(5, 5),
        strides=(2, 2),
        padding="same",
        use_bias=False,
        activation="tanh"
    )
])

# Creates the discriminator network that judges whether an image is real or generated.
discriminator = Sequential([
    # Accepts one 28-by-28 grayscale image.
    layers.Input(shape=(28, 28, 1)),

    # Extracts visual features while reducing the image dimensions.
    layers.Conv2D(
        64,
        kernel_size=(5, 5),
        strides=(2, 2),
        padding="same"
    ),

    # Applies a LeakyReLU activation.
    layers.LeakyReLU(),

    # Adds dropout to make the discriminator less likely to memorize training images.
    layers.Dropout(0.30),

    # Extracts deeper image features and reduces the dimensions again.
    layers.Conv2D(
        128,
        kernel_size=(5, 5),
        strides=(2, 2),
        padding="same"
    ),

    # Applies another LeakyReLU activation.
    layers.LeakyReLU(),

    # Adds another dropout layer.
    layers.Dropout(0.30),

    # Flattens the learned image features into one vector.
    layers.Flatten(),

    # Produces one score indicating whether the image appears real or generated.
    layers.Dense(1)
])

# Creates a binary cross-entropy loss function for GAN training.
cross_entropy = keras.losses.BinaryCrossentropy(from_logits=True)

# Defines the generator loss function.
def generator_loss(fake_output):
    # Rewards the generator when generated images are classified as real.
    return cross_entropy(
        tf.ones_like(fake_output),
        fake_output
    )

# Defines the discriminator loss function.
def discriminator_loss(real_output, fake_output):
    # Measures how well real images are classified as real.
    real_loss = cross_entropy(
        tf.ones_like(real_output),
        real_output
    )

    # Measures how well generated images are classified as fake.
    fake_loss = cross_entropy(
        tf.zeros_like(fake_output),
        fake_output
    )

    # Combines the discriminator's real-image and fake-image losses.
    return real_loss + fake_loss

# Creates an Adam optimizer for updating the generator.
generator_optimizer = keras.optimizers.Adam(
    learning_rate=0.0002,
    beta_1=0.5
)

# Creates a separate Adam optimizer for updating the discriminator.
discriminator_optimizer = keras.optimizers.Adam(
    learning_rate=0.0002,
    beta_1=0.5
)

# Marks the function for faster TensorFlow graph execution.
@tf.function
def train_dcgan_step(real_images):
    # Creates random-noise vectors for the current batch.
    random_noise = tf.random.normal(
        [tf.shape(real_images)[0], noise_dimension]
    )

    # Records generator calculations so TensorFlow can calculate gradients.
    with tf.GradientTape() as generator_tape:
        # Records discriminator calculations so TensorFlow can calculate gradients.
        with tf.GradientTape() as discriminator_tape:
            # Uses the generator to create synthetic images.
            generated_images = generator(
                random_noise,
                training=True
            )

            # Uses the discriminator to score real MNIST images.
            real_output = discriminator(
                real_images,
                training=True
            )

            # Uses the discriminator to score generated images.
            fake_output = discriminator(
                generated_images,
                training=True
            )

            # Calculates the generator's loss.
            current_generator_loss = generator_loss(fake_output)

            # Calculates the discriminator's loss.
            current_discriminator_loss = discriminator_loss(
                real_output,
                fake_output
            )

    # Calculates gradients for the generator's trainable variables.
    generator_gradients = generator_tape.gradient(
        current_generator_loss,
        generator.trainable_variables
    )

    # Calculates gradients for the discriminator's trainable variables.
    discriminator_gradients = discriminator_tape.gradient(
        current_discriminator_loss,
        discriminator.trainable_variables
    )

    # Updates the generator's trainable variables.
    generator_optimizer.apply_gradients(
        zip(
            generator_gradients,
            generator.trainable_variables
        )
    )

    # Updates the discriminator's trainable variables.
    discriminator_optimizer.apply_gradients(
        zip(
            discriminator_gradients,
            discriminator.trainable_variables
        )
    )

    # Returns both losses so they are available if needed later.
    return current_generator_loss, current_discriminator_loss

# Sets a small number of epochs for a classroom demonstration.
dcgan_epochs = 3

# Repeats the training process for the selected number of epochs.
for epoch in range(dcgan_epochs):
    # Processes every batch of MNIST images in the dataset.
    for real_image_batch in dcgan_dataset:
        # Trains the generator and discriminator on the current batch.
        generator_batch_loss, discriminator_batch_loss = train_dcgan_step(
            real_image_batch
        )

# Creates random noise for generating a small group of synthetic images.
final_noise = tf.random.normal([16, noise_dimension])

# Uses the trained generator to create synthetic MNIST-style digit images.
generated_mnist_images = generator(
    final_noise,
    training=False
)
